# Practical Flow Matching: Galaxy Generation with All the Tricks

One unified framework, many knobs. We progressively enable tricks and watch sample quality improve:

| Run | What changes | Knobs turned on |
|-----|-------------|----------------|
| 1 | Baseline | SimpleCNN, white noise, uniform time |
| 2 | Better architecture | **U-Net** |
| 3 | Training tricks | + **EMA**, **logit-normal time**, **sigma_min** |
| 4 | Smarter coupling | + **mini-batch OT**, **better base distribution** |
| 5 | Conditioning | + **class labels**, **classifier-free guidance** |

Then: class-conditional generation with guidance, and latent flow matching at 128×128 trained end-to-end.

Uses JAX + Flax + optax throughout. ~2.5 min total training on an A100.

In [ ]:
import os
from dataclasses import dataclass, field

import flax.linen as nn
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import optax
from PIL import Image
from scipy.optimize import linear_sum_assignment

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.size"] = 12

print(f"JAX devices: {jax.devices()}")

In [ ]:
%pip install -q galaxy-datasets jax jaxlib flax optax scipy

---
## 1. Data: 64x64 Galaxy Images

GalaxyMNIST (Walmsley et al.) — DESI Legacy Survey galaxies, 4 morphology classes.

In [ ]:
DATA_FILE = "data/galaxy_64.npz"
IMG_SIZE = 64
N_CLASSES = 4
CLASS_NAMES = ["smooth round", "smooth cigar", "edge-on disk", "spiral"]

if not os.path.exists(DATA_FILE):
    print("Downloading GalaxyMNIST (one-time, ~90 MB)...")
    from galaxy_datasets import galaxy_mnist

    cat_train, _ = galaxy_mnist(root="/tmp/galaxy_mnist", download=True, train=True)
    cat_test, _ = galaxy_mnist(root="/tmp/galaxy_mnist", download=True, train=False)

    def load_images(catalog, size=IMG_SIZE):
        imgs, labels = [], []
        for _, row in catalog.iterrows():
            img = Image.open(row["file_loc"]).convert("RGB").resize((size, size), Image.LANCZOS)
            imgs.append(np.array(img, dtype=np.uint8))
            labels.append(row["label"])
        return np.array(imgs), np.array(labels, dtype=np.int32)

    X_train, y_train = load_images(cat_train)
    X_test, y_test = load_images(cat_test)
    os.makedirs("data", exist_ok=True)
    np.savez_compressed(DATA_FILE, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)
    print(f"Saved to {DATA_FILE}")

data = np.load(DATA_FILE)
X_train = data["X_train"].astype(np.float32) / 255.0
y_train = data["y_train"]
X_test = data["X_test"].astype(np.float32) / 255.0
y_test = data["y_test"]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Classes: {dict(zip(CLASS_NAMES, [int((y_train == c).sum()) for c in range(4)]))}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i * 50])
    ax.set_title(CLASS_NAMES[y_train[i * 50]], fontsize=9)
    ax.axis("off")
plt.suptitle("Galaxy Zoo: 64x64 RGB", fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. The Toolkit

All building blocks defined here. The training function below takes flags to toggle each one.

### Architectures

In [ ]:
def sinusoidal_embedding(t, dim=128):
    """Map continuous t in [0,1] to a vector of sines and cosines."""
    half = dim // 2
    freqs = jnp.exp(-jnp.log(10000.0) * jnp.arange(half) / half)
    args = t[:, None] * freqs[None, :]
    return jnp.concatenate([jnp.sin(args), jnp.cos(args)], axis=-1)


class SimpleCNN(nn.Module):
    """Minimal velocity predictor. ~200K params. No skip connections."""

    n_classes: int = 0

    @nn.compact
    def __call__(self, x_t, t, y=None):
        te = nn.relu(nn.Dense(128)(sinusoidal_embedding(t, 128)))
        if self.n_classes > 0 and y is not None:
            te = te + nn.Embed(self.n_classes + 1, 128)(y)
        h = x_t
        for ch in [64, 128, 128, 64]:
            h = nn.Conv(ch, (3, 3))(h)
            h = nn.GroupNorm(8)(h)
            h = h + nn.Dense(ch)(te)[:, None, None, :]
            h = nn.silu(h)
        return nn.Conv(x_t.shape[-1], (3, 3))(h)


class ResBlock(nn.Module):
    """Residual block with FiLM time conditioning."""

    channels: int

    @nn.compact
    def __call__(self, x, t_emb):
        h = nn.silu(nn.GroupNorm(min(8, self.channels))(x))
        h = nn.Conv(self.channels, (3, 3))(h)
        # FiLM: scale and shift from time embedding
        scale = nn.Dense(self.channels)(nn.silu(t_emb))[:, None, None, :]
        shift = nn.Dense(self.channels)(nn.silu(t_emb))[:, None, None, :]
        h = h * (1 + scale) + shift
        h = nn.silu(nn.GroupNorm(min(8, self.channels))(h))
        h = nn.Conv(self.channels, (3, 3))(h)
        if x.shape[-1] != self.channels:
            x = nn.Conv(self.channels, (1, 1))(x)
        return x + h


class UNet(nn.Module):
    """U-Net velocity predictor. Resolution: 64->32->16->8->16->32->64. ~6M params."""

    channels: tuple = (48, 96, 192)
    n_classes: int = 0

    @nn.compact
    def __call__(self, x_t, t, y=None):
        C = self.channels
        te = nn.Dense(256)(nn.silu(nn.Dense(256)(sinusoidal_embedding(t, 128))))
        if self.n_classes > 0 and y is not None:
            te = te + nn.Embed(self.n_classes + 1, 256)(y)
        # Encoder
        h = nn.Conv(C[0], (3, 3))(x_t)
        h1 = ResBlock(C[0])(ResBlock(C[0])(h, te), te)  # 64x64
        h2 = ResBlock(C[1])(ResBlock(C[1])(nn.Conv(C[1], (3, 3), strides=(2, 2))(h1), te), te)  # 32x32
        h3 = ResBlock(C[2])(ResBlock(C[2])(nn.Conv(C[2], (3, 3), strides=(2, 2))(h2), te), te)  # 16x16
        # Bottleneck
        h = ResBlock(C[2])(ResBlock(C[2])(nn.Conv(C[2], (3, 3), strides=(2, 2))(h3), te), te)  # 8x8
        # Decoder + skip connections
        h = ResBlock(C[2])(
            jnp.concatenate([jax.image.resize(h, (*h3.shape[:3], h.shape[-1]), method="nearest"), h3], axis=-1), te
        )
        h = ResBlock(C[1])(
            jnp.concatenate([jax.image.resize(h, (*h2.shape[:3], h.shape[-1]), method="nearest"), h2], axis=-1), te
        )
        h = ResBlock(C[0])(
            jnp.concatenate([jax.image.resize(h, (*h1.shape[:3], h.shape[-1]), method="nearest"), h1], axis=-1), te
        )
        return nn.Conv(x_t.shape[-1], (3, 3))(nn.silu(nn.GroupNorm(8)(h)))


# Quick check
for name, M in [("SimpleCNN", SimpleCNN()), ("UNet", UNet())]:
    p = M.init(jr.PRNGKey(0), jnp.ones((1, 64, 64, 3)), jnp.array([0.5]))
    n = sum(x.size for x in jax.tree.leaves(p))
    print(f"{name}: {n:,} params")

### Base distributions

Flow matching (unlike diffusion) lets you choose **any** base distribution. If $p_0$ already resembles the data, paths are shorter and the velocity field is easier to learn.

- **White noise**: standard $\mathcal{N}(0, I)$
- **Power spectrum noise**: white noise colored to match the data's spatial frequency content (low-freq dominated for galaxies)
- **Gaussian blob noise**: noise modulated by the average radial brightness profile (bright center, dark edges)

In [ ]:
# ── Precompute data statistics for informed bases ──────────


def compute_sqrt_power_spectrum(images):
    """Average amplitude spectrum, normalized to preserve variance."""
    ft = np.fft.fft2(images, axes=(1, 2))
    sp = np.sqrt(np.mean(np.abs(ft) ** 2, axis=(0, 3)))
    return sp / np.sqrt(np.mean(sp**2))


def compute_radial_envelope(images):
    """Average radial brightness profile as a 2D envelope."""
    H, W = images.shape[1], images.shape[2]
    y, x = np.mgrid[:H, :W] - np.array([H / 2, W / 2])[:, None, None]
    r = np.sqrt(x**2 + y**2)
    avg = images.mean(axis=(0, 3))
    bins = np.linspace(0, r.max(), 33)
    profile = np.array([avg[(r >= bins[i]) & (r < bins[i + 1])].mean() for i in range(32)])
    envelope = np.interp(r, 0.5 * (bins[:-1] + bins[1:]), profile)
    return envelope / (envelope.max() + 1e-8)


SQRT_POWER = jnp.array(compute_sqrt_power_spectrum(X_train))
RADIAL_ENV = jnp.array(compute_radial_envelope(X_train))


def sample_base(key, shape, base="white"):
    """Sample from chosen base distribution."""
    if base == "power_spectrum":
        white_ft = jnp.fft.fft2(jr.normal(key, shape), axes=(1, 2))
        return jnp.fft.ifft2(white_ft * SQRT_POWER[None, :, :, None], axes=(1, 2)).real
    elif base == "blob":
        return jr.normal(key, shape) * RADIAL_ENV[None, :, :, None]
    else:
        return jr.normal(key, shape)


# Visualize the three bases
fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
for row, (label, base) in enumerate(
    [("White noise", "white"), ("Power spectrum", "power_spectrum"), ("Gaussian blob", "blob")]
):
    samples = sample_base(jr.PRNGKey(row), (4, 64, 64, 3), base)
    for col in range(4):
        img = np.array(samples[col])
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=12)
plt.suptitle("Base distributions", fontsize=14)
plt.tight_layout()
plt.show()

### Other tricks: EMA, time sampling, OT

In [ ]:
def ema_update(ema_params, params, decay=0.999):
    return jax.tree.map(lambda e, p: decay * e + (1 - decay) * p, ema_params, params)


def sample_time(key, batch_size, logit_normal=False):
    if logit_normal:
        return jax.nn.sigmoid(jr.normal(key, (batch_size,)))
    return jr.uniform(key, (batch_size,))


def ot_permutation(x0, x1):
    """OT pairing via Hungarian algorithm on 8x8 downsampled images.

    Computing pairwise distances on full 64x64x3 images is expensive.
    Downsampling to 8x8 (64x cheaper) preserves the spatial structure
    that matters for OT pairing while keeping the cost negligible.
    """
    B = x0.shape[0]
    # Downsample: average pool 64x64 -> 8x8 (8x8 stride, non-overlapping blocks)
    x0_small = x0.reshape(B, 8, 8, 8, 8, -1).mean(axis=(2, 4)).reshape(B, -1)
    x1_small = x1.reshape(B, 8, 8, 8, 8, -1).mean(axis=(2, 4)).reshape(B, -1)
    cost = np.sum((x0_small[:, None] - x1_small[None, :]) ** 2, axis=-1)
    _, perm = linear_sum_assignment(cost)
    return perm

---
## 3. Unified Training & Sampling

One function to rule them all. Every trick is a keyword argument.

In [ ]:
@dataclass
class Config:
    """All the knobs."""

    # Architecture
    use_unet: bool = False
    unet_channels: tuple = (48, 96, 192)  # U-Net channel widths per level
    # Training tricks
    use_ema: bool = False
    ema_decay: float = 0.999
    logit_normal_time: bool = False
    sigma_min: float = 0.0  # 0 = no floor; 1e-4 = standard
    use_ot: bool = False
    use_bf16: bool = False  # mixed precision (bf16 forward, f32 optimizer) — ~2x on GPU
    # Base distribution
    base_dist: str = "white"  # "white", "power_spectrum", "blob"
    # Conditioning
    n_classes: int = 0  # 0 = unconditional
    cfg_drop_prob: float = 0.1
    # Optimization
    lr: float = 3e-4
    grad_clip: float = 0.0  # 0 = no clipping
    weight_decay: float = 0.0
    cosine_schedule: bool = False
    # Training
    n_epochs: int = 50
    batch_size: int = 128
    seed: int = 0

In [ ]:
def train_fm(X_train, y_train, cfg: Config):
    """Train a flow matching model with the given config. Returns (model, params, ema_params, history)."""
    key = jr.PRNGKey(cfg.seed)
    key, init_key = jr.split(key)

    # Model
    if cfg.use_unet:
        model = UNet(channels=cfg.unet_channels, n_classes=cfg.n_classes)
    else:
        model = SimpleCNN(n_classes=cfg.n_classes)

    X = jnp.array(X_train)
    Y = jnp.array(y_train)
    dummy_y = jnp.array([0]) if cfg.n_classes > 0 else None
    params = model.init(init_key, X[:1], jnp.array([0.5]), *([dummy_y] if dummy_y is not None else []))
    ema_params = params
    n_params = sum(p.size for p in jax.tree.leaves(params))

    # Optimizer
    n_steps = cfg.n_epochs * (len(X) // cfg.batch_size)
    lr = optax.cosine_decay_schedule(cfg.lr, n_steps) if cfg.cosine_schedule else cfg.lr
    chain = []
    if cfg.grad_clip > 0:
        chain.append(optax.clip_by_global_norm(cfg.grad_clip))
    chain.append(optax.adamw(lr, weight_decay=cfg.weight_decay))
    optimizer = optax.chain(*chain)
    opt_state = optimizer.init(params)

    # Mixed precision helper
    to_bf16 = lambda x: jax.tree.map(lambda a: a.astype(jnp.bfloat16), x) if cfg.use_bf16 else x

    # Loss function (closed over config)
    def loss_fn(params, x1, y, key):
        B = x1.shape[0]
        k1, k2, k3 = jr.split(key, 3)
        x0 = sample_base(k1, x1.shape, cfg.base_dist)
        t = sample_time(k2, B, cfg.logit_normal_time)
        t4 = t[:, None, None, None]
        sm = cfg.sigma_min
        x_t = (1 - (1 - sm) * t4) * x0 + t4 * x1
        target = x1 - (1 - sm) * x0
        # bf16: cast inputs for forward pass, loss computed in f32
        x_t_cast = to_bf16(x_t)
        if cfg.n_classes > 0:
            drop = jr.bernoulli(k3, cfg.cfg_drop_prob, (B,))
            y_cond = jnp.where(drop, cfg.n_classes, y)
            pred = model.apply(to_bf16(params), x_t_cast, t, y_cond).astype(jnp.float32)
        else:
            pred = model.apply(to_bf16(params), x_t_cast, t).astype(jnp.float32)
        return jnp.mean((pred - target) ** 2)

    @jax.jit
    def step(params, opt_state, x1, y, key):
        loss, grads = jax.value_and_grad(loss_fn)(params, x1, y, key)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt_state, loss

    # Training loop
    rng = np.random.default_rng(cfg.seed)
    history = []
    desc_parts = []
    if cfg.use_ema: desc_parts.append("EMA")
    if cfg.logit_normal_time: desc_parts.append("logit-normal")
    if cfg.use_ot: desc_parts.append("OT")
    if cfg.base_dist != "white": desc_parts.append(f"base={cfg.base_dist}")
    if cfg.n_classes > 0: desc_parts.append("CFG")
    if cfg.grad_clip > 0: desc_parts.append("clip")
    if cfg.use_bf16: desc_parts.append("bf16")
    print(f"Training: {type(model).__name__} | {n_params:,} params | {cfg.n_epochs} epochs")
    if desc_parts:
        print(f"  Tricks: {', '.join(desc_parts)}")

    for epoch in range(cfg.n_epochs):
        idx = rng.permutation(len(X))
        ep_losses = []
        for start in range(0, len(X), cfg.batch_size):
            bi = idx[start : start + cfg.batch_size]
            x1, y = X[bi], Y[bi]
            if cfg.use_ot:
                key, ok = jr.split(key)
                x0_np = np.array(sample_base(ok, x1.shape, cfg.base_dist))
                perm = ot_permutation(x0_np, np.array(x1))
                x1, y = x1[perm], y[perm]
            key, sk = jr.split(key)
            params, opt_state, loss = step(params, opt_state, x1, y, sk)
            if cfg.use_ema:
                ema_params = ema_update(ema_params, params, cfg.ema_decay)
            ep_losses.append(float(loss))
        avg = np.mean(ep_losses)
        history.append(avg)
        if (epoch + 1) % max(1, cfg.n_epochs // 4) == 0 or epoch == 0:
            print(f"  Epoch {epoch + 1:3d}/{cfg.n_epochs} — loss: {avg:.4f}")

    return model, params, ema_params, history

In [ ]:
def sample_fm(model, params, cfg, key, n_samples=16, steps=100, y=None, guidance_w=1.0):
    """Generate samples via Euler integration using jax.lax.scan (compiled loop, no Python overhead)."""
    dt = 1.0 / steps
    x0 = sample_base(key, (n_samples, 64, 64, 3), cfg.base_dist)

    def euler_step(x, i):
        t = jnp.full((n_samples,), i * dt)
        if cfg.n_classes > 0 and y is not None and guidance_w != 1.0:
            unc = jnp.full((n_samples,), cfg.n_classes)
            v_u = model.apply(params, x, t, unc)
            v_c = model.apply(params, x, t, y)
            v = v_u + guidance_w * (v_c - v_u)
        elif cfg.n_classes > 0 and y is not None:
            v = model.apply(params, x, t, y)
        else:
            v = model.apply(params, x, t)
        return x + dt * v, None  # carry, output (we don't need intermediates)

    x_final, _ = jax.lax.scan(euler_step, x0, jnp.arange(steps))
    return x_final


def show_samples(samples, title="", nrow=2, ncol=8):
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.8, nrow * 1.8))
    for i, ax in enumerate(axes.flat):
        if i < len(samples):
            ax.imshow(np.clip(np.array(samples[i]), 0, 1))
        ax.axis("off")
    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

---
## 4. Progressive Experiments

Each run enables one more set of tricks. We store all results for comparison at the end.

In [ ]:
results = {}  # name -> (model, params, ema_params, history, cfg)

### Run 1: Bare-bones baseline

SimpleCNN, white noise base, uniform time sampling. No tricks.

In [ ]:
cfg1 = Config(n_epochs=50)
model, params, ema, hist = train_fm(X_train, y_train, cfg1)
results["1. Baseline"] = (model, params, ema, hist, cfg1)

show_samples(sample_fm(model, params, cfg1, jr.PRNGKey(0)), "Run 1: SimpleCNN baseline")

### Run 2: U-Net architecture

Same vanilla setup, but swap the simple CNN for a U-Net with skip connections and FiLM time conditioning. We use a smaller U-Net here (`channels=(32, 64, 128)`, ~2.8M params) — enough to show the architectural improvement.

In [ ]:
cfg2 = Config(use_unet=True, unet_channels=(32, 64, 128), n_epochs=50)
model, params, ema, hist = train_fm(X_train, y_train, cfg2)
results["2. U-Net"] = (model, params, ema, hist, cfg2)

show_samples(sample_fm(model, params, cfg2, jr.PRNGKey(0)), "Run 2: U-Net (no other tricks)")

### Run 3: Training tricks

Add **EMA**, **logit-normal time sampling**, **sigma_min**, gradient clipping, and cosine LR.

In [ ]:
cfg3 = Config(
    use_unet=True,
    use_ema=True,
    logit_normal_time=True,
    sigma_min=1e-4,
    grad_clip=1.0,
    weight_decay=1e-4,
    cosine_schedule=True,
    use_bf16=True,
    n_epochs=50,
)
model, params, ema, hist = train_fm(X_train, y_train, cfg3)
results["3. + EMA/logit-normal"] = (model, params, ema, hist, cfg3)

show_samples(sample_fm(model, params, cfg3, jr.PRNGKey(0)), "Run 3 (raw weights)")
show_samples(sample_fm(model, ema, cfg3, jr.PRNGKey(0)), "Run 3 (EMA weights)")

### Run 4: Smarter coupling

Add **mini-batch OT** pairing and switch to **power spectrum** base distribution. Paths become shorter and straighter.

In [ ]:
cfg4 = Config(
    use_unet=True,
    use_ema=True,
    logit_normal_time=True,
    sigma_min=1e-4,
    grad_clip=1.0,
    weight_decay=1e-4,
    cosine_schedule=True,
    use_bf16=True,
    use_ot=True,
    base_dist="power_spectrum",
    n_epochs=50,
)
model, params, ema, hist = train_fm(X_train, y_train, cfg4)
results["4. + OT/power spec"] = (model, params, ema, hist, cfg4)

show_samples(sample_fm(model, ema, cfg4, jr.PRNGKey(0)), "Run 4: + OT + power spectrum base (EMA)")

### Run 5: Full model with class conditioning

Add **class conditioning** and **classifier-free guidance** (15% label dropout). Train longer since this is the final model.

In [ ]:
cfg5 = Config(
    use_unet=True,
    use_ema=True,
    logit_normal_time=True,
    sigma_min=1e-4,
    grad_clip=1.0,
    weight_decay=1e-4,
    cosine_schedule=True,
    use_bf16=True,
    use_ot=True,
    base_dist="power_spectrum",
    n_classes=N_CLASSES,
    cfg_drop_prob=0.15,
    n_epochs=100,
)
model, params, ema, hist = train_fm(X_train, y_train, cfg5)
results["5. Full (+ CFG)"] = (model, params, ema, hist, cfg5)

# Unconditional samples
uncond = jnp.full((16,), N_CLASSES)
show_samples(sample_fm(model, ema, cfg5, jr.PRNGKey(0), y=uncond), "Run 5: Full model (unconditional, EMA)")

Class-conditional generation.

In [ ]:
m5, _, ema5, _, c5 = results["5. Full (+ CFG)"]

# Each morphology class
fig, axes = plt.subplots(4, 8, figsize=(14, 8))
for cls in range(4):
    y = jnp.full((8,), cls)
    s = sample_fm(m5, ema5, c5, jr.PRNGKey(cls), n_samples=8, y=y, guidance_w=2.0)
    for j in range(8):
        axes[cls, j].imshow(np.clip(np.array(s[j]), 0, 1))
        axes[cls, j].axis("off")
    axes[cls, 0].set_ylabel(CLASS_NAMES[cls], fontsize=11, rotation=90, labelpad=15)
plt.suptitle("Class-conditional generation (guidance w=2.0)", fontsize=14)
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

---
## 5. Comparison: What Helped?

All five runs side by side — same random seed, same sampling steps.

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(10, 4))
for name, (_, _, _, hist, _) in results.items():
    ax.plot(hist, label=name)
ax.set(xlabel="Epoch", ylabel="CFM Loss", title="Training loss comparison")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Sample grids side by side
n_show = 8
fig, axes = plt.subplots(len(results), n_show, figsize=(n_show * 1.8, len(results) * 1.8))

for row, (name, (m, p, ema_p, _, c)) in enumerate(results.items()):
    use_params = ema_p if c.use_ema else p
    y_arg = jnp.full((n_show,), N_CLASSES) if c.n_classes > 0 else None
    samples = sample_fm(m, use_params, c, jr.PRNGKey(42), n_samples=n_show, y=y_arg)
    for col in range(n_show):
        axes[row, col].imshow(np.clip(np.array(samples[col]), 0, 1))
        axes[row, col].axis("off")
    # Label each row with the run name
    axes[row, 0].text(
        -0.1, 0.5, name, transform=axes[row, 0].transAxes, fontsize=10, va="center", ha="right", fontweight="bold"
    )

plt.suptitle("Sample quality comparison across runs", fontsize=14)
plt.tight_layout()
plt.subplots_adjust(left=0.12)
plt.show()

---
## 6. Guidance

In [ ]:
# Guidance weight sweep — how guidance strength affects sample quality
ws = [0.0, 0.5, 1.0, 2.0, 4.0, 7.0]
target_cls = 3  # spiral

fig, axes = plt.subplots(len(ws), 6, figsize=(10, len(ws) * 1.7))
for i, w in enumerate(ws):
    y = jnp.full((6,), target_cls)
    s = sample_fm(m5, ema5, c5, jr.PRNGKey(42), n_samples=6, y=y, guidance_w=w)
    for j in range(6):
        axes[i, j].imshow(np.clip(np.array(s[j]), 0, 1))
        axes[i, j].axis("off")
    axes[i, 0].text(-0.15, 0.5, f"w = {w}", transform=axes[i, 0].transAxes, fontsize=11, va="center", ha="right")
plt.suptitle(
    f"Guidance weight sweep — class: {CLASS_NAMES[target_cls]}\n"
    f"w=0: unconditional | w=1: standard | w>1: amplified conditioning",
    fontsize=12,
)
plt.tight_layout()
plt.subplots_adjust(left=0.1)
plt.show()

---
## 7. Real vs Generated

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
rng = np.random.default_rng(0)
for i in range(8):
    axes[0, i].imshow(X_test[rng.choice(len(X_test))])
    axes[0, i].axis("off")
gen = sample_fm(m5, ema5, c5, jr.PRNGKey(99), n_samples=8, y=jnp.full((8,), N_CLASSES))
for i in range(8):
    axes[1, i].imshow(np.clip(np.array(gen[i]), 0, 1))
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Real", fontsize=12, rotation=90, labelpad=10)
axes[1, 0].set_ylabel("Generated", fontsize=12, rotation=90, labelpad=10)
plt.suptitle("Real vs Generated (64x64)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Latent Flow Matching (128×128) — End-to-End

At 64×64, pixel-space flow matching works fine. At 128×128, the U-Net becomes 4× more expensive.

**Latent flow matching** solves this: an encoder compresses images to a small latent space, flow matching operates there, and a decoder maps back to pixels. The key insight: since the CFM loss doesn't involve an ODE solve (just a forward pass), **gradients flow straight through the interpolation back into the encoder**. We can train the whole pipeline end-to-end with a single combined loss:

$$\mathcal{L} = \underbrace{\|x - \text{dec}(\text{enc}(x))\|^2}_{\text{reconstruction}} + \lambda \underbrace{\|v_\theta(z_t, t) - (z_1 - z_0)\|^2}_{\text{flow matching}}$$

where $z_1 = \text{enc}(x_1)$. No separate pretraining needed.

In [ ]:
# ── Load 128x128 data ────────────────────────────────────────

DATA_128 = "data/galaxy_128.npz"

if not os.path.exists(DATA_128):
    print("Preparing 128x128 dataset...")
    from galaxy_datasets import galaxy_mnist

    cat_train, _ = galaxy_mnist(root="/tmp/galaxy_mnist", download=True, train=True)
    cat_test, _ = galaxy_mnist(root="/tmp/galaxy_mnist", download=True, train=False)

    def load_128(catalog):
        imgs, labels = [], []
        for _, row in catalog.iterrows():
            img = Image.open(row["file_loc"]).convert("RGB").resize((128, 128), Image.LANCZOS)
            imgs.append(np.array(img, dtype=np.uint8))
            labels.append(row["label"])
        return np.array(imgs), np.array(labels, dtype=np.int32)

    X128_tr, y128_tr = load_128(cat_train)
    X128_te, y128_te = load_128(cat_test)
    os.makedirs("data", exist_ok=True)
    np.savez_compressed(DATA_128, X_train=X128_tr, y_train=y128_tr, X_test=X128_te, y_test=y128_te)

d128 = np.load(DATA_128)
X128_train = d128["X_train"].astype(np.float32) / 255.0
X128_test = d128["X_test"].astype(np.float32) / 255.0
print(f"128x128 data: train {X128_train.shape}, test {X128_test.shape}")

In [ ]:
# ── Encoder / Decoder / Latent U-Net ─────────────────────────


class Encoder(nn.Module):
    latent_dim: int = 4

    @nn.compact
    def __call__(self, x):
        for ch in [32, 64, 128]:  # 128 -> 64 -> 32 -> 16
            x = nn.silu(nn.GroupNorm(min(8, ch))(nn.Conv(ch, (3, 3), strides=(2, 2))(x)))
            x = nn.silu(nn.GroupNorm(min(8, ch))(nn.Conv(ch, (3, 3))(x)))
        return nn.Conv(self.latent_dim, (1, 1))(x)  # (B, 16, 16, 4)


class Decoder(nn.Module):
    @nn.compact
    def __call__(self, z):
        h = z
        for ch in [128, 64, 32]:  # 16 -> 32 -> 64 -> 128
            h = jax.image.resize(h, (h.shape[0], h.shape[1] * 2, h.shape[2] * 2, h.shape[3]), method="nearest")
            h = nn.silu(nn.GroupNorm(min(8, ch))(nn.Conv(ch, (3, 3))(h)))
            h = nn.silu(nn.GroupNorm(min(8, ch))(nn.Conv(ch, (3, 3))(h)))
        return nn.sigmoid(nn.Conv(3, (3, 3))(h))


class LatentUNet(nn.Module):
    """Small U-Net for 16x16 latent codes. 16->8->4->8->16."""

    channels: tuple = (64, 128)

    @nn.compact
    def __call__(self, x_t, t):
        C = self.channels
        te = nn.Dense(128)(nn.silu(nn.Dense(128)(sinusoidal_embedding(t, 64))))
        h = nn.Conv(C[0], (3, 3))(x_t)
        h1 = ResBlock(C[0])(ResBlock(C[0])(h, te), te)
        h2 = ResBlock(C[1])(ResBlock(C[1])(nn.Conv(C[1], (3, 3), strides=(2, 2))(h1), te), te)
        h = ResBlock(C[1])(ResBlock(C[1])(nn.Conv(C[1], (3, 3), strides=(2, 2))(h2), te), te)
        h = ResBlock(C[1])(
            jnp.concatenate([jax.image.resize(h, (*h2.shape[:3], h.shape[-1]), method="nearest"), h2], axis=-1), te
        )
        h = ResBlock(C[0])(
            jnp.concatenate([jax.image.resize(h, (*h1.shape[:3], h.shape[-1]), method="nearest"), h1], axis=-1), te
        )
        return nn.Conv(x_t.shape[-1], (3, 3))(nn.silu(nn.GroupNorm(8)(h)))


class LatentFM(nn.Module):
    """End-to-end: encoder + flow matching U-Net + decoder, all differentiable."""

    latent_dim: int = 4

    @nn.compact
    def __call__(self, x, x_t_latent, t):
        # Encode -> reconstruct (for recon loss)
        z = Encoder(self.latent_dim)(x)
        recon = Decoder()(z)
        # Velocity prediction in latent space (for FM loss)
        v_pred = LatentUNet()(x_t_latent, t)
        return recon, z, v_pred


# ── End-to-end training ──────────────────────────────────────


def train_latent_fm_e2e(X, n_epochs=150, batch_size=64, lr=1e-3, fm_weight=1.0, seed=0):
    key = jr.PRNGKey(seed)
    key, init_key = jr.split(key)

    model = LatentFM()
    X_jax = jnp.array(X)
    dummy_x = X_jax[:1]
    dummy_zt = jnp.zeros((1, 16, 16, 4))
    params = model.init(init_key, dummy_x, dummy_zt, jnp.array([0.5]))
    ema_params = params
    n_p = sum(p.size for p in jax.tree.leaves(params))
    print(f"Latent FM (e2e): {n_p:,} params")

    optimizer = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(lr))
    opt_state = optimizer.init(params)

    def loss_fn(params, x_batch, key):
        B = x_batch.shape[0]
        k1, k2 = jr.split(key)

        # Encode data to get z1 (target latent)
        z1 = Encoder(latent_dim=4).apply({"params": params["params"]["Encoder_0"]}, x_batch)

        # Sample noise in latent space and interpolate
        z0 = jr.normal(k1, z1.shape)
        t = jax.nn.sigmoid(jr.normal(k2, (B,)))
        t4 = t[:, None, None, None]
        sm = 1e-4
        z_t = (1 - (1 - sm) * t4) * z0 + t4 * z1
        target_v = z1 - (1 - sm) * z0

        # Forward pass: recon + velocity prediction
        recon, _, v_pred = model.apply(params, x_batch, z_t, t)

        recon_loss = jnp.mean((x_batch - recon) ** 2)
        fm_loss = jnp.mean((v_pred - target_v) ** 2)
        return recon_loss + fm_weight * fm_loss, (recon_loss, fm_loss)

    @jax.jit
    def step(params, opt_state, x_batch, key):
        (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, x_batch, key)
        updates, new_opt = optimizer.update(grads, opt_state)
        return optax.apply_updates(params, updates), new_opt, aux

    rng = np.random.default_rng(seed)
    hist_recon, hist_fm = [], []
    for epoch in range(n_epochs):
        idx = rng.permutation(len(X_jax))
        ep_r, ep_f = [], []
        for start in range(0, len(X_jax), batch_size):
            key, sk = jr.split(key)
            params, opt_state, (rl, fl) = step(params, opt_state, X_jax[idx[start : start + batch_size]], sk)
            ema_params = ema_update(ema_params, params, 0.999)
            ep_r.append(float(rl))
            ep_f.append(float(fl))
        hist_recon.append(np.mean(ep_r))
        hist_fm.append(np.mean(ep_f))
        if (epoch + 1) % max(1, n_epochs // 4) == 0 or epoch == 0:
            print(f"  Epoch {epoch + 1:3d}/{n_epochs} — recon: {hist_recon[-1]:.5f}  fm: {hist_fm[-1]:.4f}")

    return model, params, ema_params, hist_recon, hist_fm


lat_model, lat_params, lat_ema, h_r, h_f = train_latent_fm_e2e(X128_train)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(h_r)
ax1.set(xlabel="Epoch", ylabel="Recon MSE", title="Reconstruction loss")
ax1.grid(True, alpha=0.3)
ax2.plot(h_f)
ax2.set(xlabel="Epoch", ylabel="FM loss", title="Flow matching loss")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Generate: sample latents → decode to 128x128 ─────────────


def sample_latent(params, key, n=16, steps=100):
    dt = 1.0 / steps
    z = jr.normal(key, (n, 16, 16, 4))
    for i in range(steps):
        t = jnp.full((n,), i * dt)
        v = LatentUNet().apply({"params": params["params"]["LatentUNet_0"]}, z, t)
        z = z + dt * v
    return z


@jax.jit
def decode(params, z):
    return Decoder().apply({"params": params["params"]["Decoder_0"]}, z)


z_gen = sample_latent(lat_ema, jr.PRNGKey(42))
imgs_gen = decode(lat_ema, z_gen)

# Reconstructions (quality ceiling)
recon_test, _, _ = lat_model.apply(lat_ema, jnp.array(X128_test[:8]), jnp.zeros((8, 16, 16, 4)), jnp.zeros(8))

fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))
for i in range(8):
    axes[0, i].imshow(X128_test[i])
    axes[1, i].imshow(np.clip(np.array(imgs_gen[i]), 0, 1))
    axes[2, i].imshow(np.clip(np.array(recon_test[0][i] if isinstance(recon_test, tuple) else recon_test[i]), 0, 1))
    for r in range(3):
        axes[r, i].axis("off")

axes[0, 0].text(
    -0.1, 0.5, "Real 128x128", transform=axes[0, 0].transAxes, fontsize=10, va="center", ha="right", fontweight="bold"
)
axes[1, 0].text(
    -0.1,
    0.5,
    "Generated\n(latent FM)",
    transform=axes[1, 0].transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontweight="bold",
)
axes[2, 0].text(
    -0.1,
    0.5,
    "AE recon\n(ceiling)",
    transform=axes[2, 0].transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontweight="bold",
)
plt.suptitle("End-to-end Latent Flow Matching: 128x128 galaxies", fontsize=14)
plt.tight_layout()
plt.subplots_adjust(left=0.1)
plt.show()

print(
    f"FM operates on {16 * 16 * 4:,}-dim latent vs {128 * 128 * 3:,}-dim pixels = {128 * 128 * 3 // (16 * 16 * 4)}x compression"
)